In [2]:
import pandas as pd
from pathlib import Path
import time # 用来计时的
import os # 导入 os

# --- [修改 1: data_root 路径] ---
# 您的 .ipynb 在 'notebooks' 文件夹中
# 我们需要用 '..' 来 "退" 到 'Project' 根目录, 然后再进入 'data'
data_root = Path('../data')

# 2. 创建一个空列表，用来存放所有读取到的"天"的数据
all_days_data = []

# *** 我们期望在每个文件中找到的必需列 ***
REQUIRED_COLS = {'date', 'hour', 'type'}

# 3. 记录开始时间
start_time = time.time()
print(f"开始遍历所有文件夹和 CSV 文件...")
# .resolve() 会显示绝对路径，便于您检查
print(f"根目录是: {data_root.resolve()}")

# 4. 遍历所有年份 (2013 到 2020)
for year in range(2013, 2021):
    
    # --- [修改 2: year_folder 路径] ---
    # 根据您的照片, 文件夹结构是 'data/raw/beijing_YYYY'
    # 所以我们在 data_root 和 年份 之间加上 'raw'
    year_folder = data_root / 'raw' / f'beijing_{year}'
    
    if not year_folder.is_dir():
        print(f"警告：找不到文件夹 {year_folder}，跳过。")
        continue

    print(f"\n--- 真正处理文件夹: {year_folder} ---")
    
    # *** 关键修改 1: 只查找 'beijing_all_*.csv' 文件 ***
    csv_files = list(year_folder.glob('beijing_all_*.csv'))
    print(f"找到 {len(csv_files)} 个 'beijing_all_*.csv' 文件。")

    for file_path in csv_files:
        df_day = None # 先重置
        try:
            # 6. 尝试用 'utf-8' 读取
            df_day = pd.read_csv(file_path, encoding='utf-8')
        
        except UnicodeDecodeError: # 'utf-8' 失败，尝试 'gbk'
            try:
                df_day = pd.read_csv(file_path, encoding='gbk')
            except pd.errors.EmptyDataError:
                print(f"警告：文件为空，已跳过: {file_path}")
                continue # 跳过这个空文件
            except Exception as e:
                print(f"读取 {file_path} (gbk) 失败: {e}")
                continue # 跳过这个坏文件
        
        except pd.errors.EmptyDataError: # 'utf-8' 读了，但是文件是空的
            print(f"警告：文件为空，已跳过: {file_path}")
            continue # 跳过这个空文件
            
        except Exception as e: # 'utf-8' 失败，原因未知
            print(f"读取 {file_path} (utf-8) 失败: {e}")
            continue # 跳过这个坏文件
        
        # *** 关键修改 2: 检查文件内容是否正确 ***
        if df_day is not None:
            if not REQUIRED_COLS.issubset(df_day.columns):
                # .issubset() 检查 df_day.columns 是否包含了 REQUIRED_COLS 里的所有列
                print(f"警告：文件格式不正确(可能为HTML或非'all'文件)，已跳过: {file_path}")
                continue # 跳过这个格式不正确的文件

            # 7. 只有成功读取且格式正确，才加入列表
            all_days_data.append(df_day)

# 8. 检查我们是否读取到了数据
if not all_days_data:
    print("错误：没有在 'data/raw/beijing_YYYY' 目录中读取到任何数据！请检查你的文件夹结构。")
else:
    # 9. 拼接
    print(f"\n所有文件读取完毕！正在合并...")
    df_raw = pd.concat(all_days_data, ignore_index=True)

    # 10. 记录结束时间
    end_time = time.time()
    print(f"合并完成！总共耗时: {time.time() - start_time:.2f} 秒")
    print(f"总共加载了 {len(df_raw)} 行数据。")

    # 11. 显示结果
    print("\n--- 原始合并数据 (前5行) ---")
    print(df_raw.head())
    
    print("\n--- 原始合并数据 (后5行) ---")
    print(df_raw.tail())

    print("\n--- 原始合并数据 (信息) ---")
    df_raw.info()
    
# ---------------------------------------------------------------------
# --- 下半部分：数据重塑 (Melt) ---
# ---------------------------------------------------------------------

print("\n\n--- 开始进行数据重塑... ---")

# 1. 只保留我们关心的 'PM2.5' 数据
#    （你的毕业设计是关于 PM2.5 的）
df_pm25 = df_raw[df_raw['type'] == 'PM2.5'].copy()

# 2. 创建一个 'datetime' 列
#    'date' 列是 20140101 这样的整数，'hour' 是 0-23
try:
    # pd.to_datetime 会把 20140101 转换成日期
    # pd.to_timedelta 会把 0-23 的小时数转换成时间
    date_part = pd.to_datetime(df_pm25['date'], format='%Y%m%d')
    hour_part = pd.to_timedelta(df_pm25['hour'], unit='h')
    df_pm25['datetime'] = date_part + hour_part
    print("Datetime 列创建成功。")

    # 3. 识别所有的"监测站"列
    #    这些列是除了 'date', 'hour', 'type', 'datetime' 之外的所有列
    station_columns = [col for col in df_pm25.columns if col not in ['date', 'hour', 'type', 'datetime']]
    
    # 4. 关键一步：数据重塑 (Melt)
    #    把所有"监测站"列（宽）转换成两列：'station' 和 'pm2.5'（长）
    df_long = df_pm25.melt(
        id_vars=['datetime'],    # 保持不变的列
        value_vars=station_columns, # 要转换的列
        var_name='station',      # 新的"站名"列
        value_name='pm2.5'       # 新的"数值"列
    )
    print("数据重塑 (Melt) 完成。")

    # 5. 清理和排序
    # ** 注意：您原来的代码是 .dropna() (删除NaN) **
    # 这对于时间序列不是最好的，但我们先遵从您原来的逻辑
    df_final = df_long.dropna(subset=['pm2.5']) # 删除 pm2.5 是 NaN 的行
    df_final = df_final.sort_values(by=['station', 'datetime']) # 按站名和时间排序

    # 6. 显示最终成果
    print("\n--- 最终可用的数据 (前5行) ---")
    print(df_final.head())
    
    print("\n--- 最终可用的数据 (后5行) ---")
    print(df_final.tail())

    print("\n--- 最终可用的数据 (信息) ---")
    df_final.info()
    
    # --- [修改 3: 输出文件路径] ---
    # 我们不把它保存在 'notebooks' 文件夹中
    # 而是保存在 '../data/processed/' 文件夹中
    output_dir = Path('../data/processed')
    # 确保这个文件夹存在
    os.makedirs(output_dir, exist_ok=True) 
    
    output_file_name = 'beijing_pm25_2013_2020_processed.csv'
    output_file_path = output_dir / output_file_name
    
    print(f"正在保存处理好的文件到 '{output_file_path}'...")
    df_final.to_csv(output_file_path, index=False)
    print("保存完毕！")

except KeyError as e:
    print(f"错误: 找不到列 {e}。这可能是因为你的 CSV 文件没有 'date' 或 'hour' 列。")
except Exception as e:
    print(f"处理过程中发生错误: {e}")

开始遍历所有文件夹和 CSV 文件...
根目录是: D:\Project\data

--- 真正处理文件夹: ..\data\raw\beijing_2013 ---
找到 27 个 'beijing_all_*.csv' 文件。

--- 真正处理文件夹: ..\data\raw\beijing_2014 ---
找到 359 个 'beijing_all_*.csv' 文件。

--- 真正处理文件夹: ..\data\raw\beijing_2015 ---
找到 364 个 'beijing_all_*.csv' 文件。

--- 真正处理文件夹: ..\data\raw\beijing_2016 ---
找到 366 个 'beijing_all_*.csv' 文件。
警告：文件为空，已跳过: ..\data\raw\beijing_2016\beijing_all_20161230.csv
警告：文件为空，已跳过: ..\data\raw\beijing_2016\beijing_all_20161231.csv

--- 真正处理文件夹: ..\data\raw\beijing_2017 ---
找到 365 个 'beijing_all_*.csv' 文件。
警告：文件格式不正确(可能为HTML或非'all'文件)，已跳过: ..\data\raw\beijing_2017\beijing_all_20170519.csv
警告：文件格式不正确(可能为HTML或非'all'文件)，已跳过: ..\data\raw\beijing_2017\beijing_all_20170520.csv
警告：文件格式不正确(可能为HTML或非'all'文件)，已跳过: ..\data\raw\beijing_2017\beijing_all_20170521.csv
警告：文件格式不正确(可能为HTML或非'all'文件)，已跳过: ..\data\raw\beijing_2017\beijing_all_20170522.csv
警告：文件格式不正确(可能为HTML或非'all'文件)，已跳过: ..\data\raw\beijing_2017\beijing_all_20170523.csv
警告：文件格式不正确(可能为HTML或非'all'文件)，已跳过

In [ ]:
1.	ARIMA (自回归积分滑动平均模型):
o	原理： 经典统计学模型。它是一种“单变量”模型，只看 pm2.5 自身。
o	思路： “用历史的自己预测未来的自己”。它假设未来的 $pm2.5$ 浓度，可以通过它过去的值（自回归 $AR$）和过去的预测误差（滑动平均 $MA$）的线性组合来预测。
o	特点： 擅长捕捉数据自身的线性趋势和周期性（比如24小时周期）。
2.	Prophet (Facebook “先知”模型):
o	原理： 也是一种统计模型，但更现代化、更全自动。
o	思路： 它把时间序列分解为三个部分的加和：趋势 (Trend) + 周期性 (Seasonality) + 节假日效应 (Holidays)。
	趋势： 捕捉数据长期的上升或下降（比如空气治理带来的逐年下降）。
	周期性： 自动识别年周期（冬季高、夏季低）、周周期（工作日高、周末低）和日周期（早晚高峰高）。
o	特点： 对缺失数据和异常值不敏感，调参简单，非常适合快速出具“基准”预测。
3.	LSTM (长短期记忆网络):
o	原理： 深度学习模型，是一种循环神经网络 (RNN) 的变种。
o	思路： 这是一种“多变量”模型。它不像 ARIMA 只看 pm2.5 自己，而是可以同时查看多个输入（比如 $pm2.5$、温度、风速、气压、湿度）。
o	特点： 它的“记忆门”结构使其非常擅长学习时间上的长期依赖关系。例如，它能学到“3天前的持续高湿 + 1天前的风速下降，会导致今天的 $pm2.5$ 暴涨”这种复杂的非线性模式。这是 ARIMA 无法做到的。

